# GDD-ENS ensemble audit (real-model-style)

Membership-inference attack against a **faithful 10-MLP GDD-ENS ensemble** (architecture from
Supplementary Table S6), as opposed to the single-MLP proxy in `gdd_main.ipynb`. The member /
non-member split **reproduces the paper's deterministic `split_data.py`** (80/20,
`random_state=0`), so the membership ground truth is the real one without needing Supplementary
Table S1.

**One shadow = one ensemble:** each shadow model is itself a full 10-MLP ensemble, so total
trainings = `10 x (1 + num_shadow_models) x epochs` MLPs. The shipped configs use the paper-faithful
recipe (8 shadows, 200 epochs) — a **long GPU run**; lower `train.epochs` and `audit_ensemble.yaml`'s
`num_shadow_models` for a quick smoke run.

**Prerequisite:** place the GDD full feature table `msk_solid_heme_ft.csv` in `./data/` (see the
README). The first run reads it (~860 MB) and caches the prepared population.

**Deviations from the paper (documented in the README):** no oversampling and no per-member
stratified folds (target and shadows are trained identically for clean LiRA/RMIA calibration). The
target is always trained from scratch; the released GDD-ENS weights are not used (they ship as a
fragile torch-1.4 whole-model pickle).

In [ ]:
import os
import sys

import yaml

project_root = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.insert(0, project_root)

In [ ]:
import pickle

import numpy as np

from gdd_data_handler import GddDataHandler
from utils.gdd_paper_data import prepare_gdd_paper_population

with open('train_config_ensemble.yaml', 'r') as file:
    train_config = yaml.safe_load(file)

data_dir = train_config['data']['data_dir']
dataset_name = train_config['data']['dataset']
ft_path = os.path.join(data_dir, train_config['data']['feature_table'])
population_path = os.path.join(data_dir, 'gdd_ens_ensemble.pkl')
meta_cache = os.path.join(data_dir, 'gdd_ens_ensemble_meta.npz')

# Cache the prepared population so re-runs skip the ~860 MB CSV read. The .pkl is the UserDataset
# LeakPro loads (target.data_path); the .npz holds the ground-truth membership indices + class count.
if os.path.exists(population_path) and os.path.exists(meta_cache):
    with open(population_path, 'rb') as f:
        population = pickle.load(f)
    cache = np.load(meta_cache)
    member_indices, nonmember_indices = cache['member_indices'], cache['nonmember_indices']
    n_types = int(cache['n_types'])
    features, targets = population.data, population.targets
else:
    pop = prepare_gdd_paper_population(ft_path=ft_path)
    features, targets = pop.features, pop.targets
    member_indices, nonmember_indices = pop.member_indices, pop.nonmember_indices
    n_types = len(pop.encoder.classes_)
    population = GddDataHandler.UserDataset(features, targets)
    os.makedirs(data_dir, exist_ok=True)
    with open(population_path, 'wb') as f:
        pickle.dump(population, f)
    np.savez(meta_cache, member_indices=member_indices,
             nonmember_indices=nonmember_indices, n_types=n_types)

n_features = features.shape[1]
print(f'Population: {len(population)} | members: {len(member_indices)} | '
      f'non-members: {len(nonmember_indices)} | features: {n_features} | classes: {n_types}')

In [ ]:
from torch.utils.data import DataLoader

# The target trains on the true members and is evaluated on the true non-members. These index sets
# are recorded in the metadata as train_indices / test_indices, and LeakPro scores the attack
# against them.
batch_size = train_config['train']['batch_size']
train_indices = np.asarray(member_indices)
test_indices = np.asarray(nonmember_indices)

train_subset = GddDataHandler.UserDataset(features[train_indices], targets[train_indices])
test_subset = GddDataHandler.UserDataset(features[test_indices], targets[test_indices])
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False)
print(f'Target train (members): {len(train_subset)} | test (non-members): {len(test_subset)}')

In [ ]:
from torch import nn, optim, save

from gdd_ensemble_handler import GddEnsembleModelHandler
from utils.gdd_ensemble import GddEnsemble
from leakpro import LeakPro

epochs = train_config['train']['epochs']
log_dir = train_config['run']['log_dir']
os.makedirs(log_dir, exist_ok=True)

model = GddEnsemble(n_features=n_features, n_types=n_types)
criterion = nn.CrossEntropyLoss()
# Nominal optimizer for the metadata recipe; the handler builds per-member Adam from Table S6.
optimizer = optim.Adam(model.parameters(), lr=train_config['train']['learning_rate'],
                       weight_decay=train_config['train']['weight_decay'])

train_result = GddEnsembleModelHandler().train(
    dataloader=train_loader, model=model, criterion=criterion, optimizer=optimizer, epochs=epochs)
target_model = train_result.model
test_result = GddEnsembleModelHandler().eval(test_loader, target_model, criterion)
print(f'target train acc {train_result.metrics.accuracy:.4f} | test acc {test_result.accuracy:.4f}')

target_model.to('cpu')
with open(os.path.join(log_dir, 'target_model.pkl'), 'wb') as f:
    save(target_model.state_dict(), f)
meta_data = LeakPro.make_mia_metadata(
    train_result=train_result, optimizer=optimizer, loss_fn=criterion, dataloader=train_loader,
    test_result=test_result, epochs=epochs, train_indices=train_indices, test_indices=test_indices,
    dataset_name=dataset_name)
with open(os.path.join(log_dir, 'model_metadata.pkl'), 'wb') as f:
    pickle.dump(meta_data, f)

In [ ]:
# Each shadow is rebuilt from the metadata as a GddEnsemble (10 MLPs) and trained by the same
# handler on its in-split. Heavy: 8 shadow ensembles x 10 members x epochs.
leakpro = LeakPro(GddDataHandler, 'audit_ensemble.yaml', model_handler=GddEnsembleModelHandler)
mia_results = leakpro.run_audit(create_pdf=True)